# Kelompok 06 · Training di Google Colab

Sebelum run:
1. Upload folder `train/` (5 folder kelas) ke Google Drive: `MyDrive/train/`.
2. **Runtime → Change runtime type → T4 GPU**.

`model.pth` dan `experiments/` otomatis disalin ke `MyDrive/hasil/`.

In [ ]:
# mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive'
!ls -lh {DRIVE_DIR}/train

In [ ]:
# copy dataset dari Drive ke disk Colab
!mkdir -p /content/data
!cp -r {DRIVE_DIR}/train /content/data/
DATA = '/content/data/train'

import os
for c in sorted(os.listdir(DATA)):
    print(c, len(os.listdir(os.path.join(DATA, c))))

In [ ]:
!git clone https://github.com/beningapniprameswari/Studio-2_Kelompok-6.git /content/Studio-2_Kelompok-6
%cd /content/Studio-2_Kelompok-6
!pip install -q -r submission/requirements.txt
!mkdir -p experiments
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Baseline ResNet-18 (opsional)
Bobot masuk ke `baseline/model.pth`, submission utama tidak ketimpa.

In [ ]:
!python train_baseline.py --data {DATA} --epochs 8 2>&1 | tee experiments/01_baseline_resnet18.txt
!python evaluate.py --submission ./baseline --data ./my_val 2>&1 | tee -a experiments/01_baseline_resnet18.txt

## 2. ConvNeXt-Tiny
Train di 85% data, epoch terbaik dipilih dari Macro-F1 di `my_val/`.

In [ ]:
!python train.py --data {DATA} --epochs 10 --workers 2 2>&1 | tee experiments/03_convnext_tiny_10ep_gpu.txt

In [ ]:
# cek submission pakai evaluate.py, lalu simpan hasil ke Drive
!python evaluate.py --submission ./submission --data ./my_val 2>&1 | tee -a experiments/03_convnext_tiny_10ep_gpu.txt
!ls -lh submission/model.pth
!mkdir -p {DRIVE_DIR}/hasil
!cp submission/model.pth {DRIVE_DIR}/hasil/model.pth
!cp -r experiments {DRIVE_DIR}/hasil/
!ls -lhR {DRIVE_DIR}/hasil